# TP2/TP3 Longitudinal Analysis:

The following notebook contains code to perform multiple linear regression for canonical proportion (CP) at timepoint 2 (TP2) as a predictor of the following standardized speech assessments at timepoint 3 (TP3):

- GFTA
- EVT
- CTOPP
- SAILS

The following variables were used as covariates:

- Gender (dummy)
- Age (in months)
- Maternal education (on 1-7 scale - see paper for more detail.)

## GFTA

### Unweighted, Unscaled - [Expanded, Unscaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp2_master_final_with_clips.csv")  # adjust path as needed

# --- Coerce numeric columns ---
num_cols = [
    "GFTA_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    # Already numeric 0/1
    if pd.api.types.is_number(x):
        if x in (0, 1):
            return int(x)  # 0 = Male, 1 = Female
        return np.nan
    # String cases
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    if s == "" or s == "nan":
        return np.nan
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.dropna(subset=needed).copy()

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

# rows must be valid for both models to compare LLs fairly
valid_rows = df_clean[
    ["GFTA_Standard"] + predictors_exp
].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_rows].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    # Final guard against inf/nan
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X.loc[mask]
    y = y.loc[mask]
    return sm.OLS(y, X).fit()

# --- Fit models ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp, df_clean)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model: Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model: Demographics + CP → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    # Significance badge
    cp_sig = cp_p < 0.05
    print(f"\nCP significance: {'✅ CP is a statistically significant predictor of GFTA (p < .05).'}"
          if cp_sig else '❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n')
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print(f"\nModel comparison: {'✅ Expanded model significantly improves fit over Baseline (p < .05).'}"
      if lrt_sig else '❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).')

# --- Also print N used (final sample size) ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns ---
num_cols = [
    "GFTA_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1, require >0) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)  # keep strictly positive
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()  # normalize for stable scale
df_clean["__w__"] = w

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["GFTA_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Also print N used and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master =  "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file =     "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns (UNSCALED model uses raw values) ---
num_cols = [
    "GFTA_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["GFTA_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## EVT

### Unweighted, Unscaled - [Expanded]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp2_master_final_with_clips.csv")  # ensure this file exists

# --- Coerce numeric columns ---
num_cols = [
    "EVT_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.dropna(subset=needed).copy()

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

valid_rows = df_clean[
    ["EVT_Standard"] + predictors_exp
].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_rows].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X.loc[mask]
    y = y.loc[mask]
    return sm.OLS(y, X).fit()

# --- Fit models (EVT outcome) ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean)
evt_expanded = fit_model("EVT_Standard", predictors_exp, df_clean)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model: Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model: Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of EVT (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of EVT (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Also print N used (final sample size) ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns ---
num_cols = [
    "EVT_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1, require >0) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)  # strictly positive
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()  # normalize for stable scale
df_clean["__w__"] = w

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["EVT_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean, weights)
evt_expanded = fit_model("EVT_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    if cp_sig:
        print("\nCP significance: ✅ CP is a statistically significant predictor of EVT (p < .05).\n")
    else:
        print("\nCP significance: ❌ CP is not a statistically significant predictor of EVT (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
if lrt_sig:
    print("\nModel comparison: ✅ Expanded model significantly improves fit over Baseline (p < .05).")
else:
    print("\nModel comparison: ❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Also print N used and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master =  "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file =     "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns (UNSCALED model uses raw values) ---
num_cols = [
    "EVT_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["EVT_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean, weights)
evt_expanded = fit_model("EVT_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of EVT (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of EVT (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## CTOPP

### Unweighted, Unscaled - [Expanded]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp2_master_final_with_clips.csv")  # adjust path if needed

# --- Coerce numeric columns (UNSCALED/raw) ---
num_cols = [
    "CTOPP_Elision_Scaled", "canonical_proportion", "age_mos", "Maternal_Education_Level"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if isinstance(x, (int, float, np.integer, np.floating)) and pd.notna(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED/raw) ---
needed = ["CTOPP_Elision_Scaled", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values for fair comparison ---
cols_needed_all = ["CTOPP_Elision_Scaled"] + predictors_exp
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    # Guard against any remaining NaNs/infs
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X.loc[mask]
    y = y.loc[mask]
    return sm.OLS(y, X).fit()

# --- Fit models (OLS, unweighted) ---
ctopp_baseline = fit_model("CTOPP_Elision_Scaled", predictors_base, df_clean)
ctopp_expanded = fit_model("CTOPP_Elision_Scaled", predictors_exp,  df_clean)

# --- Likelihood ratio test (expanded vs baseline) ---
ll_base = ctopp_baseline.llf
ll_exp  = ctopp_expanded.llf
k_base  = len(ctopp_baseline.params)
k_exp   = len(ctopp_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model (OLS, unscaled): Demographics → CTOPP_Elision_Scaled")
print(ctopp_baseline.summary(), "\n")

print("Expanded Model (OLS, unscaled): Demographics + CP → CTOPP_Elision_Scaled")
print(ctopp_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in ctopp_expanded.params.index:
    cp_beta = ctopp_expanded.params[term]
    cp_se   = ctopp_expanded.bse[term]
    cp_t    = ctopp_expanded.tvalues[term]
    cp_p    = ctopp_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = ctopp_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting CTOPP_Elision_Scaled):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of CTOPP Elision (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of CTOPP Elision (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- N used ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns ---
num_cols = [
    "CTOPP_Elision_Scaled", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["CTOPP_Elision_Scaled", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1, require >0) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)  # strictly positive
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()  # normalize for stable scale
df_clean["__w__"] = w

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["CTOPP_Elision_Scaled"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
ctopp_baseline = fit_model("CTOPP_Elision_Scaled", predictors_base, df_clean, weights)
ctopp_expanded = fit_model("CTOPP_Elision_Scaled", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = ctopp_baseline.llf
ll_exp  = ctopp_expanded.llf
k_base  = len(ctopp_baseline.params)
k_exp   = len(ctopp_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → CTOPP_Elision_Scaled")
print(ctopp_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → CTOPP_Elision_Scaled")
print(ctopp_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in ctopp_expanded.params.index:
    cp_beta = ctopp_expanded.params[term]
    cp_se   = ctopp_expanded.bse[term]
    cp_t    = ctopp_expanded.tvalues[term]
    cp_p    = ctopp_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = ctopp_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting CTOPP_Elision_Scaled):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    if cp_sig:
        print("\nCP significance: ✅ CP is a statistically significant predictor of CTOPP Elision (p < .05).\n")
    else:
        print("\nCP significance: ❌ CP is not a statistically significant predictor of CTOPP Elision (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
if lrt_sig:
    print("\nModel comparison: ✅ Expanded model significantly improves fit over Baseline (p < .05).")
else:
    print("\nModel comparison: ❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Also print N used and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master =  "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file =     "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns (UNSCALED/raw) ---
num_cols = [
    "CTOPP_Elision_Scaled", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if isinstance(x, (int, float, np.integer, np.floating)) and pd.notna(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["CTOPP_Elision_Scaled", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["CTOPP_Elision_Scaled"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
ctopp_baseline = fit_model("CTOPP_Elision_Scaled", predictors_base, df_clean, weights)
ctopp_expanded = fit_model("CTOPP_Elision_Scaled", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = ctopp_baseline.llf
ll_exp  = ctopp_expanded.llf
k_base  = len(ctopp_baseline.params)
k_exp   = len(ctopp_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → CTOPP_Elision_Scaled")
print(ctopp_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → CTOPP_Elision_Scaled")
print(ctopp_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in ctopp_expanded.params.index:
    cp_beta = ctopp_expanded.params[term]
    cp_se   = ctopp_expanded.bse[term]
    cp_t    = ctopp_expanded.tvalues[term]
    cp_p    = ctopp_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = ctopp_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting CTOPP_Elision_Scaled):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of CTOPP Elision (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of CTOPP Elision (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## SAILS

### Unweighted, Unscaled - [Expanded]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp2_master_final_with_clips.csv")  # adjust path if needed

# --- Coerce numeric columns (UNSCALED/raw) ---
num_cols = [
    "SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos", "Maternal_Education_Level"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if isinstance(x, (int, float, np.integer, np.floating)) and pd.notna(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED/raw) ---
needed = ["SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values for fair comparison ---
cols_needed_all = ["SAILS_ProportionTestCorrect"] + predictors_exp
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    # Guard against any remaining NaNs/infs
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X.loc[mask]
    y = y.loc[mask]
    return sm.OLS(y, X).fit()

# --- Fit models (OLS, unweighted) ---
sails_baseline = fit_model("SAILS_ProportionTestCorrect", predictors_base, df_clean)
sails_expanded = fit_model("SAILS_ProportionTestCorrect", predictors_exp,  df_clean)

# --- Likelihood ratio test (expanded vs baseline) ---
ll_base = sails_baseline.llf
ll_exp  = sails_expanded.llf
k_base  = len(sails_baseline.params)
k_exp   = len(sails_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model (OLS, unscaled): Demographics → SAILS_ProportionTestCorrect")
print(sails_baseline.summary(), "\n")

print("Expanded Model (OLS, unscaled): Demographics + CP → SAILS_ProportionTestCorrect")
print(sails_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in sails_expanded.params.index:
    cp_beta = sails_expanded.params[term]
    cp_se   = sails_expanded.bse[term]
    cp_t    = sails_expanded.tvalues[term]
    cp_p    = sails_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = sails_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting SAILS_ProportionTestCorrect):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of SAILS (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of SAILS (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- N used ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns ---
num_cols = [
    "SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if isinstance(x, (int, float, np.integer, np.floating)) and pd.notna(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1, require >0) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)  # strictly positive
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()  # normalize for stable scale
df_clean["__w__"] = w

# --- Scale predictors (each column with its own scaler) ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit(df[["canonical_proportion"]].dropna()).transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any residual non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["SAILS_ProportionTestCorrect"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
sails_baseline = fit_model("SAILS_ProportionTestCorrect", predictors_base, df_clean, weights)
sails_expanded = fit_model("SAILS_ProportionTestCorrect", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = sails_baseline.llf
ll_exp  = sails_expanded.llf
k_base  = len(sails_baseline.params)
k_exp   = len(sails_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → SAILS_ProportionTestCorrect")
print(sails_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → SAILS_ProportionTestCorrect")
print(sails_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in sails_expanded.params.index:
    cp_beta = sails_expanded.params[term]
    cp_se   = sails_expanded.bse[term]
    cp_t    = sails_expanded.tvalues[term]
    cp_p    = sails_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = sails_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting SAILS_ProportionTestCorrect):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    if cp_sig:
        print("\nCP significance: ✅ CP is a statistically significant predictor of SAILS (p < .05).\n")
    else:
        print("\nCP significance: ❌ CP is not a statistically significant predictor of SAILS (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
if lrt_sig:
    print("\nModel comparison: ✅ Expanded model significantly improves fit over Baseline (p < .05).")
else:
    print("\nModel comparison: ❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Also print N used and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File paths (edit if needed) ===
primary_master = "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final_with_clips.csv"
fallback_master =  "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_master_final.csv"
counts_file =     "/Users/carissaott/Desktop/tp2_tp3_audio/tp2_clip_counts_by_child.csv"

# === Load data (prefer file that already has num_clips) ===
if os.path.exists(primary_master):
    df = pd.read_csv(primary_master)
else:
    df = pd.read_csv(fallback_master)
    if os.path.exists(counts_file):
        counts_df = pd.read_csv(counts_file)
        df = df.merge(counts_df, on="child_id", how="left")
    else:
        raise FileNotFoundError(
            "num_clips not found in master and counts file missing. "
            "Provide tp2_master_final_with_clips.csv or tp2_clip_counts_by_child.csv."
        )

# --- Coerce numeric columns (UNSCALED/raw) ---
num_cols = [
    "SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if isinstance(x, (int, float, np.integer, np.floating)) and pd.notna(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["SAILS_ProportionTestCorrect", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["SAILS_ProportionTestCorrect"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
sails_baseline = fit_model("SAILS_ProportionTestCorrect", predictors_base, df_clean, weights)
sails_expanded = fit_model("SAILS_ProportionTestCorrect", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = sails_baseline.llf
ll_exp  = sails_expanded.llf
k_base  = len(sails_baseline.params)
k_exp   = len(sails_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → SAILS_ProportionTestCorrect")
print(sails_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → SAILS_ProportionTestCorrect")
print(sails_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in sails_expanded.params.index:
    cp_beta = sails_expanded.params[term]
    cp_se   = sails_expanded.bse[term]
    cp_t    = sails_expanded.tvalues[term]
    cp_p    = sails_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = sails_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting SAILS_ProportionTestCorrect):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")

    cp_sig = cp_p < 0.05
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of SAILS (p < .05).\n"
          if cp_sig else
          "❌ CP is not a statistically significant predictor of SAILS (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

lrt_sig = lrt_p < 0.05
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_sig else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## Visuals

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# =========================
# Load data
# =========================
df_clean   = pd.read_csv("std_and_cp_tp2_with_sails.csv")          # predictors + num_clips + gender
df_outcome = pd.read_csv("tp2_std_scaler_outcome.csv")  # z-scored outcomes

# Make sure numeric fields are numeric
for col in ["age_mos", "Maternal_Education_Level", "canonical_proportion", "num_clips"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Robust gender dummy (0 = Male, 1 = Female)
def to_gender_dummy(x):
    try:
        if pd.notna(x) and float(x) in (0.0, 1.0):
            return int(x)
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df_clean["gender_dummy"] = df_clean["gender"].apply(to_gender_dummy)

# =========================
# Scale predictors (z)
# =========================
df_clean["age_mos_scaled"] = StandardScaler().fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = StandardScaler().fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = StandardScaler().fit_transform(
    df_clean[["canonical_proportion"]]
)

# =========================
# WLS weights & point sizes from num_clips
# =========================
df_clean["clip_weight"] = pd.to_numeric(df_clean["num_clips"], errors="coerce").fillna(0)

w = df_clean["clip_weight"].astype(float)
w = w.where(w > 0)                        # strictly positive; others become NaN
df_clean["w_norm"] = (w / w.mean())       # mean-normalized weights
point_size = df_clean["clip_weight"] * 0.035
# point_size = np.sqrt(df_clean["clip_weight"]) * 0.35  # optional: nicer dynamic range

# =========================
# Outcomes, labels, colors
# =========================
outcome_vars = [
    "GFTA_Standard_scaled",
    "EVT_Standard_scaled",
    "CTOPP_Standard_scaled",
    "SAILS_Standard_scaled",
]
y_labels = ["GFTA-2 (SS)", "EVT (SS)", "CTOPP-2 (ES)", "SAILS Proportion Correct"]
colors   = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]  # blue, orange, green, purple

# =========================
# Helper: build X_new to MATCH model.params.index robustly
# =========================
def make_X_new_like_model(model, df_ref, x_vals, x_col="canonical_proportion_scaled"):
    """
    Create a design matrix for prediction that has exactly the columns
    (and order) expected by `model.params.index`. Adds const/Intercept if needed.
    Holds covariates at their sample means.
    """
    base = {
        x_col: x_vals,
        "age_mos_scaled": df_ref["age_mos_scaled"].mean(),
        "gender_dummy": df_ref["gender_dummy"].mean(),
        "Maternal_Education_Level_scaled": df_ref["Maternal_Education_Level_scaled"].mean(),
    }
    X_new = pd.DataFrame(base)

    pcols = list(model.params.index)

    # Ensure constant column if model expects it
    if "const" in pcols and "const" not in X_new.columns:
        X_new["const"] = 1.0
    if "Intercept" in pcols and "Intercept" not in X_new.columns:
        X_new["Intercept"] = 1.0

    # Add any additional columns the model might have (unlikely here) as zeros
    for c in pcols:
        if c not in X_new.columns:
            X_new[c] = 0.0

    # Order to match the model exactly
    X_new = X_new[pcols]
    return X_new

# =========================
# Fit weighted, scaled models (one per outcome)
# y(z) ~ cp_scaled + age_scaled + gender + med_scaled
# =========================
models = []
valid_masks = []

for outcome in outcome_vars:
    if outcome not in df_outcome.columns:
        raise KeyError(f"Outcome column '{outcome}' not found in tp2_std_scaler_outcome.csv")

    valid = df_outcome[outcome].notna()
    d = pd.DataFrame({
        "y": df_outcome.loc[valid, outcome].values,
        "canonical_proportion_scaled": df_clean.loc[valid, "canonical_proportion_scaled"].values,
        "age_mos_scaled": df_clean.loc[valid, "age_mos_scaled"].values,
        "gender_dummy": df_clean.loc[valid, "gender_dummy"].values,
        "Maternal_Education_Level_scaled": df_clean.loc[valid, "Maternal_Education_Level_scaled"].values,
        "w": df_clean.loc[valid, "w_norm"].values
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if d.empty:
        models.append(None)
        valid_masks.append(valid)
        continue

    X = d[["canonical_proportion_scaled",
           "age_mos_scaled",
           "gender_dummy",
           "Maternal_Education_Level_scaled"]].copy()
    # Always add constant explicitly so the param name is "const"
    X.insert(0, "const", 1.0)

    y = d["y"].values
    wts = d["w"].values
    mask_w = np.isfinite(wts) & (wts > 0)
    X = X.loc[mask_w]
    y = y[mask_w]
    wts = wts[mask_w]

    model = sm.WLS(y, X, weights=wts).fit()
    models.append(model)
    valid_masks.append(valid)

# =========================
# Plot 2×2 panel
# =========================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()

for i, (outcome_var, model, color, y_label, valid) in enumerate(zip(outcome_vars, models, colors, y_labels, valid_masks)):
    ax = axes[i]

    # Scatter using the same validity mask used for modeling
    x_scatter = df_clean.loc[valid, "canonical_proportion_scaled"]
    y_scatter = df_outcome.loc[valid, outcome_var]
    size_scatter = point_size.loc[valid]

    ax.scatter(
        x_scatter,
        y_scatter,
        alpha=0.6,
        color=color,
        s=size_scatter,
        edgecolors="black",
        linewidths=0.3
    )

    # Regression line + 95% CI
    if model is not None and hasattr(model, "params"):
        x_min = df_clean["canonical_proportion_scaled"].min()
        x_max = df_clean["canonical_proportion_scaled"].max()
        x_vals = np.linspace(x_min, x_max, 200)

        X_new = make_X_new_like_model(model, df_clean, x_vals, "canonical_proportion_scaled")
        try:
            pred = model.get_prediction(X_new).summary_frame(alpha=0.05)
            ax.plot(x_vals, pred["mean"], color=color, linewidth=2)
            ax.fill_between(x_vals, pred["mean_ci_lower"], pred["mean_ci_upper"],
                            color=color, alpha=0.25)

            # β for CP (coefficient because y is standardized)
            beta_cp = model.params.get("canonical_proportion_scaled", np.nan)
            ax.text(0.05, 0.08, f"β = {beta_cp:.2f}", fontsize=12, color="black",
                    ha="left", va="bottom", transform=ax.transAxes, fontweight="bold")
        except Exception as e:
            print(f"Plotting failed for {outcome_var}: {e}")

    # Panel letter & labels
    ax.text(0.02, 0.96, f"({chr(65 + i)})", fontsize=14, color="black",
            ha="left", va="top", transform=ax.transAxes, fontweight="bold")
    ax.set_ylabel(y_label, fontsize=13, fontweight="bold")
    ax.set_xlabel("Canonical Proportion (Scaled)", fontsize=13, fontweight="bold")
    ax.tick_params(axis="both", labelsize=11)
    ax.axhline(0, color="#bbbbbb", linewidth=0.8)

fig.suptitle("TP2 Weighted (num_clips) Scaled Models:\nOutcome (z) ~ CP (scaled) + Age (scaled) + Gender + Maternal Ed (scaled)", y=1.02, fontsize=14)
fig.subplots_adjust(hspace=0.45, wspace=0.35)
plt.tight_layout()
plt.show()

# Optional: save
# fig.savefig("tp2_weighted_scaled_panels_numclips.png", dpi=300, bbox_inches="tight")
